In [ ]:
# DQN : Q-learning을 딥러닝 신경망으로 확장한 강화학습 알고리즘
# Q-table 대신 신경망을 사용
# 구조
"""
환경 Environment
   ->
현재 상태 state
   ->
행동별 Q값 예측
   ->
행동 action 선택
   ->
환경에 action 실행
   ->
reward, next_state, done (경험)
   ->
ReplayBuffer에 저장
   ->
랜덤하게 샘플링
   ->
Target Network로 target을 갱신
   ->
Q-Network 학습 (현재 상태를 입력 받아 각 행동의 Q값을 갱신)

*** DQN 동작 비유 ***
학생(Q-Network)이 환경에서 문제를 경험함 : Q-Network가 action을 선택해 행동하면
                                           state, action, reward, next_state, done 발생
    ->
문제은행(Replay Buffer)에 저장 : (state, action, reward, next_state, done)
    ->
문제은행에서 랜덤으로 문제를 꺼냄 : Replay Buffer에서 batch sample
    ->
정답지(Target Network)를 보고 목표값 계산 : target = reward + gammar * max(TargetNetwork(next_state))
    ->
학생 신경망을 갱신 : Q-Network 학습, loss = 현재 Q값 - target Q값


~~~ 이전 CartPole 유지하기 예제를 DQN으로 작업 ~~~
'기존 Q-table 방식'             'DQN 방식'
상태를 이산화해야 함            연속된 상태를 그대로 사용
Q-table로 Q값 예측              신경망 모델이 Q값 예측
argmax(Q[state])                argmax(model.predict(state))
Q[state][action] = r+감마...    model.fit()으로 현재상태 갱신
"""

!pip install gymnasium[classic-control]

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
import gymnasium as gym
from collections import deque
import matplotlib.pyplot as plt
import random
import numpy as np

env = gym.make("CartPole-v1")
num_actions = int(env.action_space.n)  # 에이전트가 취할 수 있는 행동의 공간. n: 가능한 행동 수 (2)
state_dim = int(env.observation_space.shape[0])  # 환경에서 반환하는 state 공간. 상태벡터 길이(상태변수 개수)
    # CartPole의 경우 상태는 연속값을 갖는 4차원 벡터: [카트 위치, 카트 속도, 막대 각도, 막대 각속도]
print(num_actions, state_dim)

# 2. DQN 모델 정의. Q-value 예측용 신경망 모델
def create_model():     # DQN 모델 구성 (Dense 2층 + linear 출력층)
    model = Sequential([
        Input(shape=(state_dim,)),        # ← 상태(state) 벡터 입력 (예: 4차원)
        Dense(64, activation='relu'),
        Dense(64, activation='relu'),
        Dense(num_actions, activation='linear')   # ← 출력층: 각 행동의 Q값 출력. softmax X
        # 이 층은 Q(s, a) 값을 출력하는 최종 층이다.
        # 출력은 각 액션에 대한 Q값을 의미한다. 예를들어 num_actions=2이면 → 출력은 [Q(s, 0), Q(s, 1)]
        # 왜 activation='linear'인가? Q값은 제한되지 않은 실수 값이어야 하기 때문이다.
    ])
    model.compile(optimizer=Adam(learning_rate=0.0005), loss='mse') # 예측 Q값과 target Q값의 차이를 최소화
    return model


model = create_model()   # Q-Netwok(Main Netwok) - 매 상태에서 Q(s, a)를 예측하고 model.fit()을 통해 가중치 계속 갱신

# Target Network (학습중에는 고정된 가중치를 유지하며, 일정 주기마다 model의 가중치를 복사함)
# 사용 목적 : Q-learning의 'moving target' 문제를 방지해 학습 안정성 향상
target_model = create_model()

# model의 현재 가중치를 target_model에 복사하는 작업
target_model.set_weights(model.get_weights()) # model과 가중치가 동일하게 초기화
# 전체 흐름 요약 (DQN 구조의 기본)
    # model : 현재 상태에서 Q(s, a)를 예측하고 학습함
    # target_model : 미래 상태 s′에서의 Q(s′, a′)를 계산할 때 사용 (고정된 네트워크)
    # set_weights() : 일정 주기마다 현재 모델의 가중치를 타겟 모델로 복사

# for i, w in enumerate(model.get_weights()):
#     print(i, '번째 : ', w)

# 하이퍼 파라미터
gammar = 0.99   # 할인율 (Discount Factor) : 미래 보상의 중요도를 결정함
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.05

# Replay buffer에서 꺼내 학습에 사용하는 샘플 수
# 작으면 학습이 불안정하고, 너무 크면 속도가 느려짐. CartPole처럼 가벼운 환경에서는 32~64 적당
batch_size = 64

# deque : 양방향 큐 자료구조로, append()와 popleft()가 빠르고 효율적이다.
# 경험 리플레이 버퍼.  (s, a, r, s', done) 경험을 저장해서 랜덤 샘플링
# 오래된 경험은 자동으로 삭제(FIFO). maxlen=5000: 약간 여유를 주어 다양한 패턴 학습 가능
memory = deque(maxlen=5000)    # 최대 5,000개의 경험만 저장. 경험 재사용, 무작위 샘플링으로 학습 안정화

episodes = 100    # 300 ~ 1000 정도 충분히 줘야함

# 타겟 Q 네트워크 갱신 주기. Q_target을 자주 바꾸면 불안정하고, 너무 늦으면 학습 속도 느림
# 5~10 에피소드마다 target_model.set_weights(model.get_weights())로 갱신하는 게 일반적
update_target_every = 5

reward_list = []

# 학습
for ep in range(episodes):
    # 에피소드 초기화. state는 현재 관측값 (4개 float: 위치, 속도, 각도, 각속도)
    state, _ = env.reset() # 에피소드 시작환경을 무작위로 초기화:카트 위치, 속도,막대 각도, 각속도 등
    total_reward = 0  # 이번 에피소드 동안 받은 보상의 총합
    done = False      # 에피소드 종료 조건 (막대가 너무 기울거나 카트가 벗어난 경우)

    # DQN의 핵심 루프이며, 한 에피소드 내에서 에이전트가 행동하고 학습하는 과정이 모두 들어 있다.
    # 에피소드가 끝날 때까지 state → action → next_state → reward 과정을 반복.
    # done : terminated or truncated로 CartPole에서 막대가 넘어진 경우나 시간이 다 되었을 때 종료한다.
    while not done:
        # 신경망은 batch 형태의 입력을 원함
        state_input = np.reshape(state, [1, state_dim]) # DQN에서 state를 신경망 입력형식에 맞게 변형

        # epsilon-greedy 행동 선택
        if np.random.rand() < epsilon:
            action = np.random.choice(num_actions)
        else:
            q_values = model.predict(state_input, verbose=0)
            action = np.argmax(q_values[0])

        # 환경 실행 : 다음 상태(next_state)와 보상(reward)을 받음
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated  # done은 환경이 종료되었는지를 나타냄

        # 실패 시 보상 페널티 적용 : 에피소드가 끝났다면 보상을 -10으로 강하게 부정
        # 막대가 안 넘어진 상태 (done=False) → reward = 1 유지 → 계속 잘하고 있으니까 "잘했어!"
        # 막대가 넘어졌거나 시간 종료 상태 (done=True) → reward = -10 → "야!" 강한 부정적 피드백
        if terminated:
            modified_reward = -10
        else:
            modified_reward = reward

        # Replay Buffer에 경험 저장: 에이전트의 (s, a, r, s', done) 경험을 저장
        # 나중에 무작위로 꺼내 학습함 → 샘플의 다양성 확보, 상관관계 제거
        memory.append((state, action, modified_reward, next_state, done))
        state = next_state
        total_reward += reward

        # Replay Buffer에서 batch 추출 후 학습
        if len(memory) >= batch_size:  # 최소 batch_size만큼 쌓여야 학습이 시작됨
            # 경험 버퍼에서 무작위로 batch_size개의 경험을 선택→ 상관관계 없는 다양한 상황 학습 가능
            minibatch = random.sample(memory, batch_size)

            # minibatch에 있는 한 step을 꺼내서 학습용 데이터로 가공
            states = np.array([sample[0] for sample in minibatch])
            actions = np.array([sample[1] for sample in minibatch])
            rewards = np.array([sample[2] for sample in minibatch])
            next_states = np.array([sample[3] for sample in minibatch])
            dones = np.array([sample[4] for sample in minibatch])

            # Main Q-Network로 현재 Q값 예측
            targets = model.predict(states, verbose=0)

            # Target Network로 다음 상태 Q값 예측
            next_q_values = target_model.predict(next_states, verbose=0)

            # 선택한 행동의 Q값만 Bellman target으로 수정
            for i in range(batch_size):
                if dones[i]:
                    targets[i][actions[i]] = rewards[i]
                else:
                    targets[i][actions[i]] = rewards[i] + gammar * np.max(next_q_values[i])

            # Main Q-Network 학습
            model.train_on_batch(states, targets)

    reward_list.append(total_reward)   # 한 에피소드 동안 받은 보상 합계를 저장

    # ε 감소 : (탐험 → 활용 전환)
    # 초반에는 탐험을 많이 하되, 학습이 진행되면 신경망이 예측한 최적 행동(활용)을 따르도록 함.
    # epsilon_min에 도달하면 활용만 하지는 않고, 5% 확률로 계속 탐험을 유지함 → 로컬 최적에 빠지지 않게
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay
        epsilon = max(epsilon, epsilon_min)

    # 타겟 모델 갱신
        # 왜 필요한가? DQN은 Q(s, a)를 업데이트할 때 target Q(s', a')를 사용하는데,
        # 이 둘이 동시에 바뀌면 학습이 불안정해진다 (moving target 문제)
        # 해결 방법 : model은 학습을 계속하지만 target_model은 일정 주기마다만 따라감
        # → 이렇게 하면 학습은 빠르고, 타겟은 안정적이라 효과적이다.
        # 주기 조정 : update_target_every = 5는 비교적 짧은 편 (빠른 적응)
        # 너무 자주 갱신하면 target 의미가 줄고, 너무 느리면 느려짐
        # update_target_every: 타겟 네트워크를 몇 에피소드마다 갱신할지 정한 간격 (예: 5)
    # 왜 이렇게 하는가? 문제: "moving target"  DQN에서 Q(s, a)를 업데이트할 때
        # 타겟 Q(s′, a′)도 동시에 바뀐다면 학습이 매우 불안정
        # 신경망이 움직이는 값을 기준으로 또 다른 값을 학습해야 하기 때문
    if ep % update_target_every == 0:
        target_model.set_weights(model.get_weights()) # model의 가중치를 target_model에 복사(동기화)

    # 학습 로그 출력 : 10 에피소드마다 현재 에피소드 보상과 ε를 출력
    # 디버깅 및 학습 경과 확인용. Reward가 증가하고 Epsilon은 감소하면 좋은 학습 상태
    if ep % 10 == 0:
        print(f'Episode {ep}: Reward = {total_reward:.1f}, Epsilon = {epsilon:.3f}')

# np.convolve() 이동 평균 구하기
# data = np.array([1,2,3,4,5])
# window_size = 3
# avg = np.convolve(data, np.ones(window_size) / window_size, mode='valid')
# print(avg)

# 보상 시각화
plt.figure(figsize=(10, 4))

# data 배열에 대해 window_size 크기의 이동 평균 계산용 함수
# 이동 평균(노이즈 제거)을 계산. 학습 보상 그래프를 더 부드럽게 시각화하기 위해 사용
    # np.ones(window_size)/window_size	[1/N, 1/N, ..., 1/N] 길이 N인 필터 (평균 필터)
    # np.convolve(data, ...)	데이터에 필터를 슬라이딩하며 평균값 계산
    # mode='valid'	경계값을 제외한 "완전한 평균만" 반환
def moving_average(data, window_size = 10):
    return np.convolve(data, np.ones(window_size) / window_size, mode='valid')

# reward_list: 각 에피소드에서 받은 총 보상 목록
# 일반적으로 200에 가까워질수록 학습 성공. 학습이 잘 되면 우상향 곡선을 그리게 됨
plt.plot(reward_list, label='Reward per episode')
plt.plot(moving_average(reward_list), label='Moving Avg(10)', color='red')
plt.xlabel('episode')
plt.ylabel('reward')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# 모델 저장
model.save('cartpole_model.keras')
print('모델 저장 완료')

In [ ]:
# 카트 에니메이션 생성
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from tensorflow import keras

# 1. 환경 및 모델 로드
env = gym.make("CartPole-v1", render_mode = None)
model = keras.models.load_model('cartpole_model.keras')
print(model)

state_dim = env.observation_space.shape[0]
num_actions = env.action_space.n

# 궤적 수집 (1회 실행)
flat_states = []
episode_labels = []

state, _ = env.reset()
done = False
ep_num = 0

while not done:
    flat_states.append(state.copy())
    episode_labels.append(ep_num)

    state_input = np.reshape(state, [1, state_dim])
    q_values = model.predict(state_input, verbose=0)
    action = np.argmax(q_values[0])
    next_state, reward, terminated, truncated, _ = env.step(action)
    state = next_state
    done = terminated or truncated

env.close()


In [ ]:

frame_count = len(flat_states)
# print('frame_count : ', frame_count)  # 4098

fig, ax = plt.subplots()
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-0.5, 1.5)
ax.set_title('Cartpole simulation')
ax.set_xlabel('Cart position')
ax.set_ylabel('height')

# 카트와 막대 표시
cart_width = 0.4
cart_height = 0.2
cart_y = 0.0
cart_rect = Rectangle((0, 0), cart_width, cart_height, color='black')
ax.add_patch(cart_rect)

pole_len = 1.0
line_list = ax.plot([],[], 'r-', lw=4)
pole_line = line_list[0]

episode_text = ax.text(0.05, 1.4, '', transform=ax.transData, fontsize=12, color='blue')

# 프레임 업데이트 함수
def updateFunc(frame):
    x = flat_states[frame][0]   # 카트 위치(x축 좌표)
    theta = flat_states[frame][2]  # 막대 각도
    ep_num = episode_labels[frame]  # 현재 프레임의 에피소드 번호
    cart_rect.set_xy((x - cart_width / 2, cart_y))   # 카트 위치 이동

    # 막대 끝 좌표
    x_start = x
    y_start = cart_y + cart_height
    x_end = x_start + pole_len * np.sin(theta)
    y_end = y_start + pole_len * np.cos(theta)
    pole_line.set_data([x_start, x_end], [y_start, y_end])

    episode_text.set_text(f'Episode : {ep_num}')
    return cart_rect, pole_line, episode_text


ani = FuncAnimation(fig, updateFunc, frames=frame_count, interval=50, repeat=False)
plt.close(fig)
display(HTML(ani.to_jshtml()))